## Load Data & Clean

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns 

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
from sklearn.metrics import (
    make_scorer,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.decomposition import KernelPCA
import matplotlib.pyplot as plt
from pandas.api.types import is_list_like

RANDOM_STATE = 42

In [ ]:
import time
notebook_start_time = time.time()

In [ ]:
df_all = pd.read_csv(
    "/home/dsemchin/data/data_ppmi_pd_with_best_model_subject_cols_with_subtype_probas.csv"
)

if "Unnamed: 0" in df_all.columns:
    df_all = df_all.drop(columns=["Unnamed: 0"])

print("Original dataframe shape:", df_all.shape)
df_all.head()


In [ ]:
df_main = df_all.copy()

df_main["NSD_STAGE"] = (
    df_main["NSD_STAGE"]
    .replace("NotNSD", np.nan)
    .replace("2b", 2)
)
df_main["NSD_STAGE"] = pd.to_numeric(df_main["NSD_STAGE"], errors="coerce")
df_main = df_main.sort_values(["subj_id", "time"]).reset_index(drop=True)

print("Cleaned dataframe shape:", df_main.shape)
print("Unique subjects:", df_main["subj_id"].nunique())


## Derive "subtype_proba_best"

In [ ]:
proba_cols = ["subtype_proba_0", "subtype_proba_1", "subtype_proba_2"]

df_main["subtype_proba_best"] = pd.NA
proba_available = df_main[proba_cols].notna().all(axis=1)

df_main.loc[proba_available, "subtype_proba_best"] = (
    df_main.loc[proba_available, proba_cols]
    .idxmax(axis=1)
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)
df_main["subtype_proba_best"] = df_main["subtype_proba_best"].astype("Int64")


## Clinical Features — Filter and Collapse Duplicates

In [ ]:
clinical_features = ["MCATOT", "PIGD_score", "TD_score"]
subtype_col = "subtype_proba_best"

req_cols = ["subj_id", "time", "model_split_best", subtype_col] + clinical_features

df_clinical_long = df_main[
    df_main["model_split_best"].isin(["train", "val"])
].copy()
df_clinical_long = df_clinical_long.dropna(subset=req_cols).copy()
df_clinical_long[subtype_col] = df_clinical_long[subtype_col].astype(int)

print("Rows before duplicate collapse:", df_clinical_long.shape[0])
print("Subjects before duplicate collapse:", df_clinical_long["subj_id"].nunique())


In [ ]:
# Keep OFF and NaN; prefer OFF over NaN for same subj_id-time
df_clin = df_clinical_long[
    df_clinical_long["PDSTATE"].isna() |
    df_clinical_long["PDSTATE"].astype(str).str.upper().eq("OFF")
].copy()

df_clin["pd_rank"] = df_clin["PDSTATE"].notna().astype(int)

df_clinical_unique = (
    df_clin
    .sort_values(["subj_id", "time", "pd_rank"], ascending=[True, True, False])
    .drop_duplicates(subset=["subj_id", "time"], keep="first")
    .drop(columns="pd_rank")
    .copy()
)
df_clinical_unique = df_clinical_unique.dropna(subset=req_cols).copy()
df_clinical_unique[subtype_col] = df_clinical_unique[subtype_col].astype(int)

print("Clinical unique shape:", df_clinical_unique.shape)
print("Unique subjects:", df_clinical_unique["subj_id"].nunique())
print("Duplicate subject-time rows:", df_clinical_unique.duplicated(subset=["subj_id", "time"]).sum())
print("\nSubtype counts:")
print(df_clinical_unique[subtype_col].value_counts().sort_index())


## Estimate Subject-Level Clinical Slopes

In [ ]:
#Check Subject-level lable stability 
label_stability = (
    df_clinical_unique
    .groupby("subj_id")[subtype_col]
    .nunique()
)

print("Unique target labels per subject:")
print(label_stability.value_counts().sort_index())

problem_subjects = label_stability[label_stability > 1].index.tolist()

print("\nSubjects with unstable target label:", len(problem_subjects))

if len(problem_subjects) > 0:
    display(
        df_clinical_unique[df_clinical_unique["subj_id"].isin(problem_subjects)]
        .sort_values(["subj_id", "time"])
    )

In [ ]:
def estimate_subject_slopes(group, features, time_col="time"):
    group = group.sort_values(time_col).copy()

    result = {
        "subj_id": group["subj_id"].iloc[0],
        "model_split_best": group["model_split_best"].iloc[0],
        subtype_col: group[subtype_col].iloc[0],
        "n_timepoints": group[time_col].nunique(),
        "first_time": group[time_col].min(),
        "last_time": group[time_col].max(),
        "followup_duration": group[time_col].max() - group[time_col].min(),
    }
    if group[time_col].nunique() < 2:
        for feature in features:
            result[f"{feature}_intercept"] = np.nan
            result[f"{feature}_slope"] = np.nan
        return pd.Series(result)

    
    X_time = group[[time_col]].values

    for feature in features:
        y_feature = group[feature].values

        model = LinearRegression()
        model.fit(X_time, y_feature)

        result[f"{feature}_intercept"] = model.intercept_
        result[f"{feature}_slope"] = model.coef_[0]

    return pd.Series(result)


df_subject = (
    df_clinical_unique
    .groupby("subj_id")
    .apply(estimate_subject_slopes, features=clinical_features)
    .reset_index(drop=True)
)
print(df_subject["n_timepoints"].value_counts().sort_index())
df_subject[subtype_col] = df_subject[subtype_col].astype(int)

# Make MoCA slope point in worsening direction
df_subject["MCATOT_worsening_slope"] = -df_subject["MCATOT_slope"]

print("Subject-level dataframe shape:", df_subject.shape)
print("Subjects:", df_subject["subj_id"].nunique())

df_subject.head()

## Model 1: Clinical Slopes 

In [ ]:
clinical_slope_cols = [
    "MCATOT_worsening_slope",
    "PIGD_score_slope",
    "TD_score_slope",
]

clinical_intercept_cols = [
    "MCATOT_intercept",
    "PIGD_score_intercept",
    "TD_score_intercept",
]

# Start with slopes only.
# comparison to slopes + intercepts.
#feature_cols = clinical_slope_cols + clinical_intercept_cols
feature_cols = clinical_slope_cols

print("Using features:")
print(feature_cols)

In [ ]:
required_model_cols = [
    "subj_id",
    "model_split_best",
    subtype_col
] + feature_cols

df_model = df_subject.copy()

df_model = df_model.replace([np.inf, -np.inf], np.nan)
df_model = df_model.dropna(subset=required_model_cols).copy()

print("Model dataframe shape:", df_model.shape)
print("Subjects used:", df_model["subj_id"].nunique())

print("\nTarget counts by split:")
print(
    pd.crosstab(
        df_model["model_split_best"],
        df_model[subtype_col],
        margins=True
    )
)

In [ ]:
df_train = df_model[
    df_model["model_split_best"] == "train"
].copy()

df_holdout = df_model[
    df_model["model_split_best"] == "val"
].copy()

X_train = df_train[feature_cols].copy()
y_train = df_train[subtype_col].astype(int).copy()

X_holdout = df_holdout[feature_cols].copy()
y_holdout = df_holdout[subtype_col].astype(int).copy()

print("X_train shape:", X_train.shape)
print("X_holdout shape:", X_holdout.shape)

print("\nTraining target counts:")
print(y_train.value_counts().sort_index())

print("\nHoldout target counts:")
print(y_holdout.value_counts().sort_index())

train_subjects = set(df_train["subj_id"])
holdout_subjects = set(df_holdout["subj_id"])

print("\nSubject overlap between train and holdout:")
print(len(train_subjects.intersection(holdout_subjects)))

In [ ]:
# Confirm existing train / holdout setup from previous cells
print("Feature columns:")
print(feature_cols)

print("\nTarget column:")
print(subtype_col)

print("\nX_train shape:", X_train.shape)
print("X_holdout shape:", X_holdout.shape)

print("\nTraining target counts:")
print(y_train.value_counts().sort_index())

print("\nHoldout/test target counts:")
print(y_holdout.value_counts().sort_index())

print("\nmodel_split_best counts in df_model:")
print(df_model["model_split_best"].value_counts())

In [ ]:
class CustomRepeatedStratifiedKFold:
    def __init__(
        self, y, stratify_by=None, n_splits=5, n_repeats=3, include_y_in_strat_labels=True, random_state=None
    ):
        self.n_splits = n_splits
        self.n_repeats = n_repeats
        self.random_state = random_state
        if stratify_by is None:
            self.strat_labels = np.array(y)
        else:
            # Make sure stratify_by is a list of lists
            if is_list_like(stratify_by) == False:
                raise ValueError("stratify_by must be a list of lists")
            if all(is_list_like(item) for item in stratify_by) == False:
                raise ValueError("stratify_by must be a list of lists")
            if include_y_in_strat_labels:
                self.strat_labels = np.array(
                    [f"{yy}_" + "_".join(str(e[i]) for e in stratify_by) for i, yy in enumerate(y)]
                )
            else:
                self.strat_labels = np.array(["_".join(str(e[i]) for e in stratify_by) for i in range(len(y))])

    def split(self, X, y=None, groups=None):
        rsk = RepeatedStratifiedKFold(n_splits=self.n_splits, n_repeats=self.n_repeats, random_state=self.random_state)
        return rsk.split(X, self.strat_labels)

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits * self.n_repeats

In [ ]:
# Concatenate train and holdout into one pool
X_all = pd.concat([X_train, X_holdout], axis=0).reset_index(drop=True)
y_all = pd.concat([y_train, y_holdout], axis=0).reset_index(drop=True)

# Build the DanTrainTestLabel: "train" for Dan's train subjects, "val" for Dan's holdout
label_all= pd.Series(
    ["train"] * len(X_train) + ["val"] * len(X_holdout)
).reset_index(drop=True)

In [ ]:
# Pipeline:
# StandardScaler -> KernelPCA -> StandardScaler -> LogisticRegression
pipe = Pipeline(steps=[
    ("scaler_before_kpca", StandardScaler()),
    ("kpca",   KernelPCA(kernel="rbf", eigen_solver= "dense", 
                         remove_zero_eig=True, random_state=RANDOM_STATE)),
    ("scaler_after_kpca", StandardScaler()), 
    ("clf",    LogisticRegression(
                   max_iter=5000,
                   tol=1e-6,
                   random_state=RANDOM_STATE,
                   solver="saga"   
               ))
])

pipe


In [ ]:
# Scoring
# Main selection score: macro F1
scoring = {
    "macro_f1": make_scorer(f1_score, average="macro"),
    "accuracy": make_scorer(accuracy_score),
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
}

# Repeated stratified K-fold objects
# Outer CV estimates performance
# Inner CV selects hyperparameters

outer_cv = CustomRepeatedStratifiedKFold(
    y=y_all,
    stratify_by=[label_all],   # joint stratification: y × DanTrainTestLabel
    n_splits=5,
    n_repeats=3,
    include_y_in_strat_labels=True,
    random_state=RANDOM_STATE
)

In [ ]:
max_kpca_components = 75

kpca_n_components_grid = np.unique(
    np.round(
        np.logspace(
            np.log10(2),
            np.log10(max_kpca_components),
            num=10
        )
    ).astype(int)
)

print(kpca_n_components_grid)

In [ ]:
# Hyperparameter grid
# GridSearchCV will search these combinations inside inner CV
param_grid = {
    "kpca__n_components": kpca_n_components_grid,
    "kpca__gamma":        [0.001, 0.01, 0.1, 1.0, 10.0],
    "clf__C":             np.logspace(-4, 3, 8),
    "clf__class_weight":  ["balanced"],
    "clf__penalty":       ["elasticnet"],
    "clf__l1_ratio":      [0, 0.01, 0.1, 0.5, 0.9, 1],
}

In [ ]:
# Nested repeated stratified CV
nested_results = []
best_params_list = []
outer_true_clinical = []
outer_pred_clinical = []
convergence_records = []

outer_fold = 0


for outer_train_idx, outer_test_idx in outer_cv.split(X_all):

    outer_fold += 1

    print("\n" + "=" * 70)
    print(f"Outer fold {outer_fold}")
    print("=" * 70)

    # --------------------------------------------------------
    # Outer split
    # --------------------------------------------------------
    X_outer_train = X_all.iloc[outer_train_idx].copy()
    X_outer_test  = X_all.iloc[outer_test_idx].copy()
    y_outer_train = y_all.iloc[outer_train_idx].copy()
    y_outer_test  = y_all.iloc[outer_test_idx].copy()

    print("Outer train shape:", X_outer_train.shape)
    print("Outer test shape:", X_outer_test.shape)

    print("\nOuter train label counts:")
    print(y_outer_train.value_counts().sort_index())

    print("\nOuter test label counts:")
    print(y_outer_test.value_counts().sort_index())
   
    # Build inner custom CV fresh for this outer fold's subjects
    # label_allsliced to only the outer-train subjects
    label_outer_train = label_all.iloc[outer_train_idx].reset_index(drop=True)
    
    # Inner GridSearchCV
    inner_cv_custom = CustomRepeatedStratifiedKFold(
        y=y_outer_train.reset_index(drop=True),
        stratify_by=[label_outer_train],
        n_splits=5,
        n_repeats=3,
        include_y_in_strat_labels=True, # combineds y and label_outer_train
        random_state=RANDOM_STATE
    )

    #Precompute inner splits as a list so GridSearchCV can consume them
    inner_splits = list(inner_cv_custom.split(X_outer_train))

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring=scoring,
        refit="macro_f1",
        cv=inner_splits,   # precomputed list, not a splitter object
        n_jobs=-1,
        return_train_score=True,
        error_score="raise",
        verbose=0
    )

    search.fit(X_outer_train, y_outer_train)
    
    best_clf = search.best_estimator_.named_steps["clf"]
    convergence_records.append({
        "outer_fold": outer_fold,
        "best_params": search.best_params_,
        "best_inner_macro_f1": search.best_score_,
        "n_iter": best_clf.n_iter_,
        "max_iter": best_clf.max_iter,
        "hit_max_iter": np.any(best_clf.n_iter_ >= best_clf.max_iter)
    })
 
    # Outer test evaluation
    y_outer_pred = search.predict(X_outer_test)
    
    outer_true_clinical.extend(y_outer_test.tolist())
    outer_pred_clinical.extend(y_outer_pred.tolist())

    inner_best_macro_f1 = search.best_score_
    outer_macro_f1 = f1_score(y_outer_test, y_outer_pred, average="macro")
    outer_accuracy = accuracy_score(y_outer_test, y_outer_pred)
    outer_balanced_accuracy = balanced_accuracy_score(y_outer_test, y_outer_pred)

    print("\nBest inner CV macro F1:")
    print(inner_best_macro_f1)

    print("\nOuter macro F1:")
    print(outer_macro_f1)

    print("\nOuter accuracy:")
    print(outer_accuracy)

    print("\nOuter balanced accuracy:")
    print(outer_balanced_accuracy)

    print("\nBest parameters:")
    print(search.best_params_)

    nested_results.append({
        "outer_fold": outer_fold,
        "inner_best_macro_f1": inner_best_macro_f1,
        "outer_macro_f1": outer_macro_f1,
        "outer_accuracy": outer_accuracy,
        "outer_balanced_accuracy": outer_balanced_accuracy,
        "best_params": search.best_params_
    })

    best_params_list.append(search.best_params_)
    
    best_clf = search.best_estimator_.named_steps["clf"]

    print("Best params:", search.best_params_)
    print("n_iter_:", best_clf.n_iter_)
    print("max_iter:", best_clf.max_iter)

In [ ]:
# Nested CV results summary
nested_results_df = pd.DataFrame(nested_results)

display(nested_results_df)

nested_summary = nested_results_df[
    [
        "inner_best_macro_f1",
        "outer_macro_f1",
        "outer_accuracy",
        "outer_balanced_accuracy"
    ]
].agg(["mean", "std", "min", "max"]).T

display(nested_summary)

print("Mean outer macro F1:")
print(nested_results_df["outer_macro_f1"].mean())

print("\nStandard deviation outer macro F1:")
print(nested_results_df["outer_macro_f1"].std())

In [ ]:
# Which hyperparameters were selected across outer folds?
best_params_df = pd.DataFrame(best_params_list)

display(best_params_df)

print("\nSelected kpca__n_components:")
print(best_params_df["kpca__n_components"].value_counts())

print("\nSelected kpca__gamma:")
print(best_params_df["kpca__gamma"].value_counts())

print("\nSelected clf__C:")
print(best_params_df["clf__C"].value_counts())

print("\nSelected clf__l1_ratio:")
print(best_params_df["clf__l1_ratio"].value_counts().sort_index())

In [ ]:
df_convergence = pd.DataFrame(convergence_records)

display(df_convergence)

print(df_convergence["hit_max_iter"].value_counts())

In [ ]:
# Plot outer-fold macro F1
plt.figure(figsize=(10, 5))

plt.plot(
    nested_results_df["outer_fold"],
    nested_results_df["outer_macro_f1"],
    marker="o"
)

plt.axhline(
    nested_results_df["outer_macro_f1"].mean(),
    linestyle="--",
    label=f"Mean outer macro F1 = {nested_results_df['outer_macro_f1'].mean():.3f}"
)

plt.xlabel("Outer fold")
plt.ylabel("Macro F1")
plt.title("Nested repeated stratified CV on training subjects")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

sns.histplot(
    data=nested_results_df,
    x="outer_macro_f1",
    bins=10,
    kde=True,
    edgecolor="black"
)

plt.axvline(
    nested_results_df["outer_macro_f1"].mean(),
    linestyle="--",
    color="black",
    label=f"Mean = {nested_results_df['outer_macro_f1'].mean():.3f}"
)

plt.xlabel("Outer-fold Macro F1")
plt.ylabel("Count / Density")
plt.title("Nested CV: Held-Out Macro F1 Distribution (Clinical Slopes)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_ncomp_counts = best_params_df["kpca__n_components"].value_counts().sort_index()

plt.figure(figsize=(8, 4))
plt.bar(best_ncomp_counts.index, best_ncomp_counts.values, edgecolor="black", alpha=0.7)
plt.plot(best_ncomp_counts.index, best_ncomp_counts.values, marker="o", linewidth=2)
plt.xlabel("Selected kpca__n_components")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best n_components Selection Distribution")
plt.xticks(best_ncomp_counts.index)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_gamma_counts = best_params_df["kpca__gamma"].value_counts().sort_index()
x_pos = np.arange(len(best_gamma_counts))

plt.figure(figsize=(8, 4))
plt.bar(x_pos, best_gamma_counts.values, edgecolor="black", alpha=0.7)
plt.plot(x_pos, best_gamma_counts.values, marker="o", linewidth=2)
plt.xticks(
    ticks=x_pos,
    labels=[f"{g:.4f}" for g in best_gamma_counts.index],
    rotation=45,
    ha="right"
)
plt.xlabel("Selected kpca__gamma")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best Gamma Selection Distribution")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_C_counts = best_params_df["clf__C"].value_counts().sort_index()
x_pos = np.arange(len(best_C_counts))

plt.figure(figsize=(8, 4))
plt.bar(x_pos, best_C_counts.values, edgecolor="black", alpha=0.7)
plt.plot(x_pos, best_C_counts.values, marker="o", linewidth=2)
plt.xticks(
    ticks=x_pos,
    labels=[str(c) for c in best_C_counts.index],
    rotation=45,
    ha="right"
)
plt.xlabel("Selected clf__C")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best C Selection Distribution")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_l1_ratio_counts = (
    best_params_df["clf__l1_ratio"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(8, 4))

plt.bar(
    best_l1_ratio_counts.index.astype(str),
    best_l1_ratio_counts.values,
    edgecolor="black",
    alpha=0.7
)

plt.plot(
    best_l1_ratio_counts.index.astype(str),
    best_l1_ratio_counts.values,
    marker="o",
    linewidth=2
)

plt.xlabel("Selected clf__l1_ratio")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best Elastic Net l1_ratio Selection Distribution")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("Expected pooled predictions:", len(y_all) * outer_cv.n_repeats)
print("Actual pooled predictions:", len(outer_true_clinical))
print("True/pred lengths match:", len(outer_true_clinical) == len(outer_pred_clinical))

labels = sorted(y_all.unique())

cm = confusion_matrix(
    outer_true_clinical,
    outer_pred_clinical,
    labels=labels   
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

disp.plot(cmap="Blues", values_format="d")
plt.title("Clinical Model: Pooled Outer-Fold Confusion Matrix")
plt.show()

print(classification_report(
    outer_true_clinical,
    outer_pred_clinical,
    labels=labels,
    zero_division=0 
))

## Model 2: Residualized Imaging Slopes  

In [ ]:
cortical_resid_features = [
    col for col in df_main.columns
    if col.startswith(("L_", "R_")) and col.endswith("_thickavg_resid")
]

subcortical_resid_features = [
    col for col in [
        "LLatVent_resid", "RLatVent_resid",
        "Lthal_resid",    "Rthal_resid",
        "Lcaud_resid",    "Rcaud_resid",
        "Lput_resid",     "Rput_resid",
        "Lpal_resid",     "Rpal_resid",
        "Lhippo_resid",   "Rhippo_resid",
        "Lamyg_resid",    "Ramyg_resid",
        "Laccumb_resid",  "Raccumb_resid",
    ]
    if col in df_main.columns
]

resid_features = cortical_resid_features + subcortical_resid_features

print(f"Cortical: {len(cortical_resid_features)}  "
      f"Subcortical: {len(subcortical_resid_features)}  "
      f"Total: {len(resid_features)}")


In [ ]:
# Build residualized imaging slopes
req_resid_cols = [
    "subj_id",
    "time",
    "model_split_best",
    subtype_col,
] + resid_features

df_resid_slope_input = (
    df_clinical_unique[req_resid_cols]
    .dropna(subset=req_resid_cols)
    .copy()
)

df_resid_slope_input = (
    df_resid_slope_input
    .groupby("subj_id")
    .filter(lambda g: g["time"].nunique() >= 2)
    .copy()
)

print("Residual slope input shape:", df_resid_slope_input.shape)
print("Residual slope input subjects:", df_resid_slope_input["subj_id"].nunique())
print("Duplicate subject-time rows:", df_resid_slope_input.duplicated(["subj_id", "time"]).sum())

In [ ]:
df_resid_slopes = (
    df_resid_slope_input
    .groupby("subj_id")
    .apply(
        estimate_subject_slopes,
        features=resid_features
    )
    .reset_index(drop=True)
)

df_resid_slopes[subtype_col] = df_resid_slopes[subtype_col].astype(int)

resid_slope_cols = [
    f"{feature}_slope"
    for feature in resid_features
]

resid_intercept_cols = [
    f"{feature}_intercept"
    for feature in resid_features
]

print("Residual slopes dataframe shape:", df_resid_slopes.shape)
print("Number of residual slope features:", len(resid_slope_cols))
print("Number of residual intercept features:", len(resid_intercept_cols))

df_resid_slopes.head()

**Residualized Imaging Slopes Only, Build X & Y**

In [ ]:
df_model_check_resid = df_resid_slopes.copy()

df_model_check_resid = df_model_check_resid[
    df_model_check_resid["model_split_best"].isin(["train", "val"])
].copy()

required_resid_cols = [
    "subj_id",
    "model_split_best",
    subtype_col,
] + resid_slope_cols

df_model_check_resid = df_model_check_resid.replace([np.inf, -np.inf], np.nan)

df_model_resid = (
    df_model_check_resid
    .dropna(subset=required_resid_cols)
    .copy()
)

print("Residual model dataframe shape:", df_model_resid.shape)
print("Residual model subjects:", df_model_resid["subj_id"].nunique())

print("\nTarget counts by split:")
print(
    pd.crosstab(
        df_model_resid["model_split_best"],
        df_model_resid[subtype_col],
        margins=True
    )
)

In [ ]:
df_resid_train = df_model_resid[
    df_model_resid["model_split_best"] == "train"
].copy()

df_resid_holdout = df_model_resid[
    df_model_resid["model_split_best"] == "val"
].copy()

X_resid_train = df_resid_train[resid_slope_cols].copy()
y_resid_train = df_resid_train[subtype_col].astype(int).copy()

X_resid_holdout = df_resid_holdout[resid_slope_cols].copy()
y_resid_holdout = df_resid_holdout[subtype_col].astype(int).copy()

print("X_resid_train shape:", X_resid_train.shape)
print("X_resid_holdout shape:", X_resid_holdout.shape)

print("\nResidual training target counts:")
print(y_resid_train.value_counts().sort_index())

print("\nResidual holdout target counts:")
print(y_resid_holdout.value_counts().sort_index())

print("\nSubject overlap between train and holdout:")
print(len(set(df_resid_train["subj_id"]).intersection(set(df_resid_holdout["subj_id"]))))

In [ ]:
print("X_resid_train shape:", X_resid_train.shape)

print("Any NaN:", X_resid_train.isna().any().any())
print("Any inf:", np.isinf(X_resid_train.to_numpy()).any())

print("Number of constant columns:")
print((X_resid_train.std(axis=0) == 0).sum())

In [ ]:
# Concatenate train and holdout into one pool
X_resid_all = pd.concat([X_resid_train, X_resid_holdout], axis=0).reset_index(drop=True)
y_resid_all = pd.concat([y_resid_train, y_resid_holdout], axis=0).reset_index(drop=True)

# Build the DanTrainTestLabel: "train" for Dan's train subjects, "val" for Dan's holdout
label_resid = pd.Series(
    ["train"] * len(X_resid_train) + ["val"] * len(X_resid_holdout)
).reset_index(drop=True)

In [ ]:
# Pipeline:
# StandardScaler -> KernelPCA -> StandardScaler -> LogisticRegression
pipe = Pipeline(steps=[
    ("scaler_before_kpca", StandardScaler()),
    ("kpca",   KernelPCA(kernel="rbf", eigen_solver= "randomized", 
                         iterated_power=7, remove_zero_eig=True, 
                         random_state=RANDOM_STATE)),
    ("scaler_after_kpca", StandardScaler()), 
    ("clf",    LogisticRegression(
                   max_iter=10000,
                   tol= 1e-6, 
                   random_state=RANDOM_STATE,
                   solver="saga"   # supports both l1 and l2
               ))
])

pipe

In [ ]:
# outer fold train size ≈ X_resid_all * (4/5)
# inner fold train size ≈ outer_train_size * (4/5)
# min_inner_train_size ≈ floor(len(X_resid_all) * 4/5) * 4/5) - 1

outer_train_size = len(X_resid_all) - len(X_resid_all) // 5
min_inner_train_size = outer_train_size - outer_train_size // 5
max_kpca_components = min(min_inner_train_size - 1, X_resid_all.shape[1])

print("Estimated min inner train size:", min_inner_train_size)
print("Safe max kpca_n_components:", max_kpca_components)

kpca_n_components_grid = np.unique(
    np.round(
        np.logspace(
            np.log10(2),
            np.log10(max_kpca_components),
            num=10
        )
    ).astype(int)
)

print(kpca_n_components_grid)

In [ ]:
param_grid = {
    "kpca__n_components": kpca_n_components_grid,
    "kpca__gamma":        [0.001, 0.01, 0.1, 1.0, 10],
    "clf__C":             np.logspace(-4, 3, 8),
    "clf__class_weight":  ["balanced"],
    "clf__penalty":       ["elasticnet"],
    "clf__l1_ratio":      [0, 0.01, 0.1, 0.5, 0.9, 1],
}


In [ ]:
outer_cv_resid = CustomRepeatedStratifiedKFold(
    y=y_resid_all,
    stratify_by=[label_resid],   # joint stratification: y × DanTrainTestLabel
    n_splits=5,
    n_repeats=3,
    include_y_in_strat_labels=True,
    random_state=RANDOM_STATE
)

In [ ]:
# Nested repeated stratified CV
# Residualized imaging slopes only
nested_results_resid = []
best_params_list_resid = []
convergence_records_resid = []
outer_true_resid = []
outer_pred_resid = []

outer_fold = 0


for outer_train_idx, outer_test_idx in outer_cv_resid.split(X_resid_all):

    outer_fold += 1

    print("\n" + "=" * 70)
    print(f"Outer fold {outer_fold}")
    print("=" * 70)

    # --------------------------------------------------------
    # Outer split
    # --------------------------------------------------------
    X_outer_train = X_resid_all.iloc[outer_train_idx].copy()
    X_outer_test  = X_resid_all.iloc[outer_test_idx].copy()
    y_outer_train = y_resid_all.iloc[outer_train_idx].copy()
    y_outer_test  = y_resid_all.iloc[outer_test_idx].copy()


    print("Outer train shape:", X_outer_train.shape)
    print("Outer test shape:", X_outer_test.shape)

    print("\nOuter train label counts:")
    print(y_outer_train.value_counts().sort_index())

    print("\nOuter test label counts:")
    print(y_outer_test.value_counts().sort_index())

    # Inner CV: custom joint stratification on y × Dan train/val label
    label_outer_train = label_resid.iloc[outer_train_idx].reset_index(drop=True)

    inner_cv_custom = CustomRepeatedStratifiedKFold(
        y=y_outer_train.reset_index(drop=True),
        stratify_by=[label_outer_train],
        n_splits=5,
        n_repeats=3,
        include_y_in_strat_labels=True,
        random_state=RANDOM_STATE
    )
    # Precompute inner splits as a list so GridSearchCV can consume them
    inner_splits = list(inner_cv_custom.split(X_outer_train))

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring=scoring,
        refit="macro_f1",
        cv=inner_splits,   # precomputed list, not a splitter object
        n_jobs=-1,
        return_train_score=True,
        error_score="raise",
        verbose=0
    )

    search.fit(X_outer_train, y_outer_train)
    best_clf = search.best_estimator_.named_steps["clf"]
    convergence_records_resid.append({
        "outer_fold": outer_fold,
        "best_params": search.best_params_,
        "best_inner_macro_f1": search.best_score_,
        "n_iter": best_clf.n_iter_,
        "max_iter": best_clf.max_iter,
        "hit_max_iter": np.any(best_clf.n_iter_ >= best_clf.max_iter)
    })

    # --------------------------------------------------------
    # Outer test evaluation
    # --------------------------------------------------------
    y_outer_pred = search.predict(X_outer_test)

    outer_true_resid.extend(y_outer_test.tolist())
    outer_pred_resid.extend(y_outer_pred.tolist())

    inner_best_macro_f1 = search.best_score_
    outer_macro_f1 = f1_score(y_outer_test, y_outer_pred, average="macro")
    outer_accuracy = accuracy_score(y_outer_test, y_outer_pred)
    outer_balanced_accuracy = balanced_accuracy_score(y_outer_test, y_outer_pred)

    print("\nBest inner CV macro F1:")
    print(inner_best_macro_f1)

    print("\nOuter macro F1:")
    print(outer_macro_f1)

    print("\nOuter accuracy:")
    print(outer_accuracy)

    print("\nOuter balanced accuracy:")
    print(outer_balanced_accuracy)

    print("\nBest parameters:")
    print(search.best_params_)

    nested_results_resid.append({
        "outer_fold": outer_fold,
        "inner_best_macro_f1": inner_best_macro_f1,
        "outer_macro_f1": outer_macro_f1,
        "outer_accuracy": outer_accuracy,
        "outer_balanced_accuracy": outer_balanced_accuracy,
        "best_params": search.best_params_
    })

    best_params_list_resid.append(search.best_params_)

In [ ]:
# Nested CV results summary
# Residualized imaging slopes only

nested_results_resid_df = pd.DataFrame(nested_results_resid)

display(nested_results_resid_df)

nested_summary_resid = nested_results_resid_df[
    [
        "inner_best_macro_f1",
        "outer_macro_f1",
        "outer_accuracy",
        "outer_balanced_accuracy"
    ]
].agg(["mean", "std", "min", "max"]).T

display(nested_summary_resid)

print("Mean outer macro F1:")
print(nested_results_resid_df["outer_macro_f1"].mean())

print("\nStandard deviation outer macro F1:")
print(nested_results_resid_df["outer_macro_f1"].std())

In [ ]:
best_params_resid_df = pd.DataFrame(best_params_list_resid)

display(best_params_resid_df)

print("\nSelected kpca__n_components:")
print(best_params_resid_df["kpca__n_components"].value_counts())

print("\nSelected kpca__gamma:")
print(best_params_resid_df["kpca__gamma"].value_counts())

print("\nSelected clf__C:")
print(best_params_resid_df["clf__C"].value_counts())

print("\nSelected clf__l1_ratio:")
print(best_params_resid_df["clf__l1_ratio"].value_counts().sort_index())

In [ ]:
df_convergence_resid = pd.DataFrame(convergence_records_resid)

display(df_convergence_resid)

print(df_convergence_resid["hit_max_iter"].value_counts())

In [ ]:
# Plot outer-fold macro F1
# Residualized imaging slopes only

plt.figure(figsize=(10, 5))

plt.plot(
    nested_results_resid_df["outer_fold"],
    nested_results_resid_df["outer_macro_f1"],
    marker="o"
)

plt.axhline(
    nested_results_resid_df["outer_macro_f1"].mean(),
    linestyle="--",
    label=f"Mean outer macro F1 = {nested_results_resid_df['outer_macro_f1'].mean():.3f}"
)

plt.xlabel("Outer fold")
plt.ylabel("Macro F1")
plt.title("Residualized imaging slopes: nested repeated stratified CV")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Macro F1 score Plot 
plt.figure(figsize=(8, 4))

sns.histplot(
    data=nested_results_resid_df,
    x="outer_macro_f1",
    bins=10,
    kde=True,
    edgecolor="black"
)

plt.axvline(
    nested_results_resid_df["outer_macro_f1"].mean(),
    linestyle="--",
    color="black",
    label=f"Mean = {nested_results_resid_df['outer_macro_f1'].mean():.3f}"
)

plt.xlabel("Outer-fold Macro F1")
plt.ylabel("Count / Density")
plt.title("Nested CV: Held-Out Macro F1 Distribution (Residual Imaging Slopes)")
plt.legend()
plt.tight_layout()
plt.show()

#Best n Components 
best_ncomp_counts = best_params_resid_df["kpca__n_components"].value_counts().sort_index()

plt.figure(figsize=(8, 4))
plt.bar(best_ncomp_counts.index, best_ncomp_counts.values, edgecolor="black", alpha=0.7)
plt.plot(best_ncomp_counts.index, best_ncomp_counts.values, marker="o", linewidth=2)
plt.xlabel("Selected kpca__n_components")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best n_components Selection Distribution")
plt.xticks(best_ncomp_counts.index)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

#Best Gamma 
best_gamma_counts = best_params_resid_df["kpca__gamma"].value_counts().sort_index()
x_pos = np.arange(len(best_gamma_counts))

plt.figure(figsize=(8, 4))
plt.bar(x_pos, best_gamma_counts.values, edgecolor="black", alpha=0.7)
plt.plot(x_pos, best_gamma_counts.values, marker="o", linewidth=2)
plt.xticks(
    ticks=x_pos,
    labels=[f"{g:.4f}" for g in best_gamma_counts.index],
    rotation=45,
    ha="right"
)
plt.xlabel("Selected kpca__gamma")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best Gamma Selection Distribution")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

#Best C 
best_C_counts = best_params_resid_df["clf__C"].value_counts().sort_index()
x_pos = np.arange(len(best_C_counts))

plt.figure(figsize=(8, 4))
plt.bar(x_pos, best_C_counts.values, edgecolor="black", alpha=0.7)
plt.plot(x_pos, best_C_counts.values, marker="o", linewidth=2)
plt.xticks(
    ticks=x_pos,
    labels=[str(c) for c in best_C_counts.index],
    rotation=45,
    ha="right"
)
plt.xlabel("Selected clf__C")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best C Selection Distribution")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

#Best L1 Penalty 
best_l1_ratio_counts = (
    best_params_resid_df["clf__l1_ratio"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(8, 4))

plt.bar(
    best_l1_ratio_counts.index.astype(str),
    best_l1_ratio_counts.values,
    edgecolor="black",
    alpha=0.7
)

plt.plot(
    best_l1_ratio_counts.index.astype(str),
    best_l1_ratio_counts.values,
    marker="o",
    linewidth=2
)

plt.xlabel("Selected clf__l1_ratio")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best Elastic Net l1_ratio Selection Distribution")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("Expected pooled predictions:", len(y_resid_all) * outer_cv_resid.n_repeats)
print("Actual pooled predictions:", len(outer_true_resid))
print("True/pred lengths match:", len(outer_true_resid) == len(outer_pred_resid))
labels = sorted(y_resid_all.unique())

cm_resid = confusion_matrix(
    outer_true_resid,
    outer_pred_resid,
    labels=labels
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_resid,
    display_labels=labels
)

disp.plot(cmap="Blues", values_format="d")
plt.title("Residual Imaging Model: Pooled Outer-Fold Confusion Matrix")
plt.show()

print(classification_report(
    outer_true_resid,
    outer_pred_resid,
    labels=labels,
    zero_division=0
))

## Model 3: Clinical Slopes + Residualized Imaging Slopes

In [ ]:
# Model 3: Clinical slopes + residualized imaging slopes
df_combined_slopes = pd.merge(
    df_subject[
        ["subj_id", "model_split_best", subtype_col, "n_timepoints", "followup_duration"]
        + clinical_slope_cols
    ],
    df_resid_slopes[
        ["subj_id"] + resid_slope_cols
    ],
    on="subj_id",
    how="inner"
)

combined_feature_cols = clinical_slope_cols + resid_slope_cols

print("Combined slopes shape:", df_combined_slopes.shape)
print("Number of combined features:", len(combined_feature_cols))
print("Subjects in combined model:", df_combined_slopes["subj_id"].nunique())

df_combined_slopes.head()

In [ ]:
# Build X and y for combined model
df_model_combined = df_combined_slopes.copy()

df_model_combined = df_model_combined[
    df_model_combined["model_split_best"].isin(["train", "val"])
].copy()

required_combined_cols = [
    "subj_id",
    "model_split_best",
    subtype_col,
] + combined_feature_cols

df_model_combined = df_model_combined.replace([np.inf, -np.inf], np.nan)

df_model_combined = (
    df_model_combined
    .dropna(subset=required_combined_cols)
    .copy()
)

print("Combined model dataframe shape:", df_model_combined.shape)
print("Combined model subjects:", df_model_combined["subj_id"].nunique())

print("\nTarget counts by split:")
print(
    pd.crosstab(
        df_model_combined["model_split_best"],
        df_model_combined[subtype_col],
        margins=True
    )
)

In [ ]:
# Train / holdout split for combined model
df_combined_train = df_model_combined[
    df_model_combined["model_split_best"] == "train"
].copy()

df_combined_holdout = df_model_combined[
    df_model_combined["model_split_best"] == "val"
].copy()

X_combined_train = df_combined_train[combined_feature_cols].copy()
y_combined_train = df_combined_train[subtype_col].astype(int).copy()

X_combined_holdout = df_combined_holdout[combined_feature_cols].copy()
y_combined_holdout = df_combined_holdout[subtype_col].astype(int).copy()

print("X_combined_train shape:", X_combined_train.shape)
print("X_combined_holdout shape:", X_combined_holdout.shape)

print("\nCombined training target counts:")
print(y_combined_train.value_counts().sort_index())

print("\nCombined holdout target counts:")
print(y_combined_holdout.value_counts().sort_index())

print("\nSubject overlap between train and holdout:")
print(
    len(
        set(df_combined_train["subj_id"])
        .intersection(set(df_combined_holdout["subj_id"]))
    )
)

In [ ]:
# Concatenate train and holdout into one pool
X_combined_all = pd.concat([X_combined_train, X_combined_holdout], axis=0).reset_index(drop=True)
y_combined_all = pd.concat([y_combined_train, y_combined_holdout], axis=0).reset_index(drop=True)

# Build the DanTrainTestLabel: "train" for Dan's train subjects, "val" for Dan's holdout
label_combined = pd.Series(
    ["train"] * len(X_combined_train) + ["val"] * len(X_combined_holdout)
).reset_index(drop=True)

In [ ]:
# Pipeline:
# StandardScaler -> KernelPCA -> StandardScaler -> LogisticRegression
pipe = Pipeline(steps=[
    ("scaler_before_kpca", StandardScaler()),
    ("kpca",   KernelPCA(kernel="rbf", eigen_solver= "randomized", 
                         iterated_power=7, remove_zero_eig=True, 
                         random_state=RANDOM_STATE)),
    ("scaler_after_kpca", StandardScaler()), 
    ("clf",    LogisticRegression(
                   max_iter=20000,
                   tol= 1e-3, 
                   random_state=RANDOM_STATE,
                   solver="saga"
               ))
])

pipe

In [ ]:
# outer fold train size ≈ X_resid_all * (4/5)
# inner fold train size ≈ outer_train_size * (4/5)
# min_inner_train_size ≈ floor(len(X_resid_all) * 4/5) * 4/5) - 1

outer_train_size = len(X_combined_all) - len(X_combined_all) // 5
min_inner_train_size = outer_train_size - outer_train_size // 5
max_kpca_components = min(min_inner_train_size - 1, X_combined_all.shape[1])

print("Estimated min inner train size:", min_inner_train_size)
print("Safe max kpca_n_components:", max_kpca_components)

kpca_n_components_grid = np.unique(
    np.round(
        np.logspace(
            np.log10(2),
            np.log10(max_kpca_components),
            num=10
        )
    ).astype(int)
)

print("KPCA n_components grid:")
print(kpca_n_components_grid)

In [ ]:
param_grid = {
    "kpca__n_components": kpca_n_components_grid,
    "kpca__gamma":        [0.001, 0.01, 0.1, 1.0, 10.0],
    "clf__C":             np.logspace(-4, 3, 8),
    "clf__class_weight":  ["balanced"],
    "clf__penalty":       ["elasticnet"],
    "clf__l1_ratio":      [0, 0.01, 0.1, 0.5, 0.9, 1],
}

In [ ]:
outer_cv_combined = CustomRepeatedStratifiedKFold(
    y=y_combined_all,
    stratify_by=[label_combined],   # joint stratification: y × DanTrainTestLabel
    n_splits=5,
    n_repeats=3,
    include_y_in_strat_labels=True,
    random_state=RANDOM_STATE
)

In [ ]:
# Nested repeated stratified CV
# Clinical slopes + residualized imaging slopes
nested_results_combined = []
best_params_list_combined = []
convergence_records_combined = []
outer_true_combined = []
outer_pred_combined = []

outer_fold = 0


for outer_train_idx, outer_test_idx in outer_cv_combined.split(X_combined_all):

    outer_fold += 1

    print("\n" + "=" * 70)
    print(f"Outer fold {outer_fold}")
    print("=" * 70)

    # --------------------------------------------------------
    # Outer split
    # --------------------------------------------------------
    X_outer_train = X_combined_all.iloc[outer_train_idx].copy()
    X_outer_test  = X_combined_all.iloc[outer_test_idx].copy()
    y_outer_train = y_combined_all.iloc[outer_train_idx].copy()
    y_outer_test  = y_combined_all.iloc[outer_test_idx].copy()


    print("Outer train shape:", X_outer_train.shape)
    print("Outer test shape:", X_outer_test.shape)

    print("\nOuter train label counts:")
    print(y_outer_train.value_counts().sort_index())

    print("\nOuter test label counts:")
    print(y_outer_test.value_counts().sort_index())

    
    # Inner GridSearchCV

    label_outer_train = label_combined.iloc[outer_train_idx].reset_index(drop=True)

    inner_cv_custom = CustomRepeatedStratifiedKFold(
        y=y_outer_train.reset_index(drop=True),
        stratify_by=[label_outer_train],
        n_splits=5,
        n_repeats=3,
        include_y_in_strat_labels=True,
        random_state=RANDOM_STATE
    )

    # Precompute inner splits as a list so GridSearchCV can consume them
    inner_splits = list(inner_cv_custom.split(X_outer_train))

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring=scoring,
        refit="macro_f1",
        cv=inner_splits,   # precomputed list, not a splitter object
        n_jobs=-1,
        return_train_score=True,
        error_score="raise",
        verbose=0
    )

    search.fit(X_outer_train, y_outer_train)
    best_clf = search.best_estimator_.named_steps["clf"]
    convergence_records_combined.append({
        "outer_fold": outer_fold,
        "best_params": search.best_params_,
        "best_inner_macro_f1": search.best_score_,
        "n_iter": best_clf.n_iter_,
        "max_iter": best_clf.max_iter,
        "hit_max_iter": np.any(best_clf.n_iter_ >= best_clf.max_iter)
    })
    
    # Outer test evaluation
    y_outer_pred = search.predict(X_outer_test)

    outer_true_combined.extend(y_outer_test.tolist())
    outer_pred_combined.extend(y_outer_pred.tolist())
    
    inner_best_macro_f1 = search.best_score_
    outer_macro_f1 = f1_score(y_outer_test, y_outer_pred, average="macro")
    outer_accuracy = accuracy_score(y_outer_test, y_outer_pred)
    outer_balanced_accuracy = balanced_accuracy_score(y_outer_test, y_outer_pred)

    print("\nBest inner CV macro F1:")
    print(inner_best_macro_f1)

    print("\nOuter macro F1:")
    print(outer_macro_f1)

    print("\nOuter accuracy:")
    print(outer_accuracy)

    print("\nOuter balanced accuracy:")
    print(outer_balanced_accuracy)

    print("\nBest parameters:")
    print(search.best_params_)

    nested_results_combined.append({
        "outer_fold": outer_fold,
        "inner_best_macro_f1": inner_best_macro_f1,
        "outer_macro_f1": outer_macro_f1,
        "outer_accuracy": outer_accuracy,
        "outer_balanced_accuracy": outer_balanced_accuracy,
        "best_params": search.best_params_
    })

    best_params_list_combined.append(search.best_params_)

In [ ]:
# Nested CV results summary
# Clinical slopes + residualized imaging slopes
nested_results_combined_df = pd.DataFrame(nested_results_combined)

display(nested_results_combined_df)

nested_summary_combined = nested_results_combined_df[
    [
        "inner_best_macro_f1",
        "outer_macro_f1",
        "outer_accuracy",
        "outer_balanced_accuracy"
    ]
].agg(["mean", "std", "min", "max"]).T

display(nested_summary_combined)

print("Mean outer macro F1:")
print(nested_results_combined_df["outer_macro_f1"].mean())

print("\nStandard deviation outer macro F1:")
print(nested_results_combined_df["outer_macro_f1"].std())

**Combined Model-Selected Hyperparameters** 

In [ ]:
best_params_combined_df = pd.DataFrame(best_params_list_combined)

display(best_params_combined_df)

print("\nSelected kpca__n_components:")
print(best_params_combined_df["kpca__n_components"].value_counts())

print("\nSelected kpca__gamma:")
print(best_params_combined_df["kpca__gamma"].value_counts())

print("\nSelected clf__C:")
print(best_params_combined_df["clf__C"].value_counts())

print("\nSelected clf__l1_ratio:")
print(best_params_combined_df["clf__l1_ratio"].value_counts().sort_index())

In [ ]:
#check if best model converged
df_convergence_combined = pd.DataFrame(convergence_records_combined)

display(df_convergence_combined)

print(df_convergence_combined["hit_max_iter"].value_counts())

**Combined Model-Plot Outer Macro F1** 

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    nested_results_combined_df["outer_fold"],
    nested_results_combined_df["outer_macro_f1"],
    marker="o"
)

plt.axhline(
    nested_results_combined_df["outer_macro_f1"].mean(),
    linestyle="--",
    label=f"Mean outer macro F1 = {nested_results_combined_df['outer_macro_f1'].mean():.3f}"
)

plt.xlabel("Outer fold")
plt.ylabel("Macro F1")
plt.title("Clinical + residualized imaging slopes: nested repeated stratified CV")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Macro F1 Score 
plt.figure(figsize=(8, 4))

sns.histplot(
    data=nested_results_combined_df,
    x="outer_macro_f1",
    bins=10,
    kde=True,
    edgecolor="black"
)

plt.axvline(
    nested_results_combined_df["outer_macro_f1"].mean(),
    linestyle="--",
    color="black",
    label=f"Mean = {nested_results_combined_df['outer_macro_f1'].mean():.3f}"
)

plt.xlabel("Outer-fold Macro F1")
plt.ylabel("Count / Density")
plt.title("Nested CV: Macro F1 Distribution (Clinical Slopes + Residual Slopes)")
plt.legend()
plt.tight_layout()
plt.show()

#best n components 
best_ncomp_counts = best_params_combined_df["kpca__n_components"].value_counts().sort_index()

plt.figure(figsize=(8, 4))
plt.bar(best_ncomp_counts.index, best_ncomp_counts.values, edgecolor="black", alpha=0.7)
plt.plot(best_ncomp_counts.index, best_ncomp_counts.values, marker="o", linewidth=2)
plt.xlabel("Selected kpca__n_components")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best n_components Selection Distribution")
plt.xticks(best_ncomp_counts.index)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

#Best gamma 
best_gamma_counts = best_params_combined_df["kpca__gamma"].value_counts().sort_index()
x_pos = np.arange(len(best_gamma_counts))

plt.figure(figsize=(8, 4))
plt.bar(x_pos, best_gamma_counts.values, edgecolor="black", alpha=0.7)
plt.plot(x_pos, best_gamma_counts.values, marker="o", linewidth=2)
plt.xticks(
    ticks=x_pos,
    labels=[f"{g:.4f}" for g in best_gamma_counts.index],
    rotation=45,
    ha="right"
)
plt.xlabel("Selected kpca__gamma")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best Gamma Selection Distribution")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

#best C 
best_C_counts = best_params_combined_df["clf__C"].value_counts().sort_index()
x_pos = np.arange(len(best_C_counts))

plt.figure(figsize=(8, 4))
plt.bar(x_pos, best_C_counts.values, edgecolor="black", alpha=0.7)
plt.plot(x_pos, best_C_counts.values, marker="o", linewidth=2)
plt.xticks(
    ticks=x_pos,
    labels=[str(c) for c in best_C_counts.index],
    rotation=45,
    ha="right"
)
plt.xlabel("Selected clf__C")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best C Selection Distribution")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

#best L1 ratio
best_l1_ratio_counts = (
    best_params_combined_df["clf__l1_ratio"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(8, 4))

plt.bar(
    best_l1_ratio_counts.index.astype(str),
    best_l1_ratio_counts.values,
    edgecolor="black",
    alpha=0.7
)

plt.plot(
    best_l1_ratio_counts.index.astype(str),
    best_l1_ratio_counts.values,
    marker="o",
    linewidth=2
)

plt.xlabel("Selected clf__l1_ratio")
plt.ylabel("Number of outer folds")
plt.title("Nested CV: Best Elastic Net l1_ratio Selection Distribution")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()




In [ ]:
print("Expected pooled predictions:", len(y_combined_all) * outer_cv_combined.n_repeats)
print("Actual pooled predictions:", len(outer_true_combined))
print("True/pred lengths match:", len(outer_true_combined) == len(outer_pred_combined))

labels = sorted(y_combined_all.unique())

cm_combined = confusion_matrix(
    outer_true_combined,
    outer_pred_combined,
    labels=labels
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_combined,
    display_labels=labels
)

disp.plot(cmap="Blues", values_format="d")
plt.title("Combined Model: Pooled Outer-Fold Confusion Matrix")
plt.show()

print(classification_report(
    outer_true_combined,
    outer_pred_combined,
    labels=labels,
    zero_division=0
))

In [ ]:
notebook_elapsed = time.time() - notebook_start_time
hours = int(notebook_elapsed // 3600)
minutes = int((notebook_elapsed % 3600) // 60)
seconds = int(notebook_elapsed % 60)
print(f"Total notebook runtime: {hours}h {minutes}m {seconds}s")